In [ ]:
import argparse
import base64
import difflib
import io
import json
import os
import re
import time
from collections import defaultdict
from pathlib import Path

import pandas as pd
from openai import OpenAI
from PIL import Image

# Thai digit → Arabic digit translation table

In [ ]:
THAI_MAP = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")

# Get Datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
DATA_DIR      = Path("/content/drive/My Drive/SuperAI Engineer SS6/Week-2/data")
IMAGES_DIR    = DATA_DIR / "images"
TEMPLATE_FILE = DATA_DIR / "submission_template.csv"
CHECKPOINT    = "checkpoint_typhoon.json"
OUTPUT_FILE   = "submission_typhoon.csv"

# Typhoon Configuration

In [ ]:
BASE_URL     = "https://api.opentyphoon.ai/v1"
MODEL        = "typhoon-ocr"
MAX_TOKENS   = 4096
RATE_DELAY   = 3.5   # 20 req/min → ~3s gap
MAX_RETRIES  = 3
RETRY_DELAY  = 5

### Method : โหลดไฟล์ CSV จาก TEMPLATE_FILE เข้าเป็น pandas DataFrame (df)
- สร้าง defaultdict(list) สำหรับเก็บข้อมูลแบบจัดกลุ่ม (doc_parties)
- วน loop ทุกแถวใน DataFrame
- ใช้ doc_id เป็น key ใน dictionary
เพิ่มข้อมูลแต่ละแถวเข้าไปใน list ของ doc_id นั้น
เก็บเฉพาะ field สำคัญ:
   - id
   - row_num (แปลงเป็น int)
   - party_name
- จัดข้อมูลให้อยู่ในรูปแบบ: 1 doc → หลายรายการ (list of dict)
return ออกมา 2 อย่าง: df (ข้อมูลเดิม), doc_parties (ข้อมูลที่จัดกลุ่มแล้ว)

**ผลลัพธ์**

```
doc_parties = {
    "doc_1": [
        {"id": 1, "row_num": 1, "party_name": "A"},
        {"id": 2, "row_num": 2, "party_name": "B"},
    ],
    "doc_2": [
        {"id": 3, "row_num": 1, "party_name": "C"},
    ]
}
```

In [ ]:
def load_template():
    df = pd.read_csv(TEMPLATE_FILE)
    doc_parties = defaultdict(list)
    for _, row in df.iterrows():
        doc_parties[row["doc_id"]].append({
            "id":         row["id"],
            "row_num":    int(row["row_num"]),
            "party_name": row["party_name"],
        })
    return df, doc_parties

### Method : Get Image
- รับค่า doc_id (เช่น "0001")
- สร้าง list ว่าง images สำหรับเก็บ path ของรูป
- เช็คไฟล์หน้าแรก:
  - ชื่อไฟล์: doc_id.png เช่น 0001.png ถ้ามี → เพิ่มเข้า images
- วน loop ตั้งแต่ page 2 ถึง 9:
  - สร้างชื่อไฟล์รูปแบบ: doc_id_page{n}.png เช่น 0001_page2.png
  - เช็คว่าไฟล์มีอยู่ไหม (exists()) ถ้ามี → เพิ่มเข้า images
  - คืนค่า list ของ path รูปทั้งหมดที่หาเจอ

**ผลลัพธ์**
```
[
  Path("0001.png"),
  Path("0001_page2.png"),
  Path("0001_page3.png"),
]
```

In [ ]:
def get_doc_images(doc_id: str) -> list[Path]:
    images = []
    first = IMAGES_DIR / f"{doc_id}.png"
    if first.exists():
        images.append(first)
    for n in range(2, 10):
        p = IMAGES_DIR / f"{doc_id}_page{n}.png"
        if p.exists():
            images.append(p)
    return images

### Method : Resize ภาพให้ด้านยาวสุดไม่เกิน max_size แล้ว encode base64

- รับ input:
  - path → path ของไฟล์รูป
  - max_size → ขนาดด้านยาวสูงสุด (default = 1600 px)
- เปิดรูปภาพด้วย Image.open(path)
  - อ่านขนาดรูป (w, h)
- ตรวจสอบว่ารูปใหญ่เกินไหม:
  - ถ้า max(w, h) > max_size
    - คำนวณ scale: scale = max_size / ด้านที่ยาวที่สุด
  - resize รูปให้เล็กลง โดยรักษาอัตราส่วน (aspect ratio) ใช้ Image.LANCZOS → คุณภาพสูง
- สร้าง buffer ใน memory (BytesIO)
- save รูปลง buffer ใน format PNG
- แปลง binary → Base64:
  - base64.standard_b64encode(...)
  - .decode() → แปลงเป็น string
- return: Base64 string ของรูป

In [ ]:
def to_base64(path: Path, max_size: int = 1600) -> str:
    """Resize ภาพให้ด้านยาวสุดไม่เกิน max_size แล้ว encode base64"""
    img = Image.open(path)
    w, h = img.size
    if max(w, h) > max_size:
        scale = max_size / max(w, h)
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.standard_b64encode(buf.getvalue()).decode()

# Vote Extraction from Typhoon OCR HTML Output

1. สรุปภาพรวม (High-level)

Input:
  - raw = ข้อความจาก OCR (อาจเป็น JSON หรือ HTML)
  - party_list = list ของพรรคที่ต้องหา

Output: {party_name: votes}

```
{"ประชาชน": "34167", "เพื่อไทย": "28900"}
```

2. กรอง output แปลกๆ จาก model ถ้า raw เริ่มด้วย "Extract all text from the image." ถือว่า OCR fail → return {}

3. พยายาม parse JSON ก่อน ใช้ json.loads(raw)
   - กรณีเป็นเลขไทย เช่น `{"ประชาชน": "๓๔,๑๖๗"}`
     - แปลงเลขไทย → อารบิก (THAI_MAP)
     - ลบ comma
     - เก็บเป็น `{"ประชาชน": "34167"}`
   - กรณีเป็น JSON wrapper (มี HTML ข้างใน) `{"ส.ส.": "<table>...</table>"}`
     - ดึง value ทั้งหมดมาต่อกันเป็น text เพื่อเอาไป parse HTML ต่อ
     - ถ้า parse JSON ไม่ได้ ข้ามไป parse HTML เลย
4. Parse HTML table
   - ใช้ regex ดึง `<td>...</td>` ทั้งหมด `cells = re.findall("<td>...</td>")`
   - clean text (ลบ space เกิน)
5. หา "พรรค → คะแนน" โดย loop ทุกพรรคใน party_list ใน `for p in party_list`
   - หา cell ที่มีชื่อพรรค
6. หาเลขคะแนนใกล้ๆ
   - เมื่อเจอชื่อพรรค: มอง cell ปัจจุบัน + 2 cell ถัดไป จะได้ `cells[i], cells[i+1], cells[i+2]`
   - ใช้ regex หาเลข:
     - รองรับเลขไทย ๐-๙
     - รองรับเลขอารบิก 0-9
7. แปลงเลข
   - แปลงเลขไทย → อารบิก (THAI_MAP)
   - ลบ comma เช่น "34,167" → "34167"
8. validate
   - ต้องเป็นตัวเลข และ ≥ 10 (กัน noise เช่น 0, 1)
9. เก็บผลลัพธ์ `votes[party] = num`
10. return ผลลัพธ์ ได้ dict พรรค → คะแนน

In [ ]:
def parse_votes_from_ocr(raw: str, party_list: list[dict]) -> dict[str, str]:
    """Parse Typhoon OCR output (HTML table or JSON wrapper) to extract party→votes.

    Typhoon OCR returns the document as an HTML table wrapped in a JSON object,
    e.g. {"ส.ส.": "<table>...<td>ประชาชน</td><td>๓๔,๑๖๗ ...</td>..."}
    We extract <td> cells and find party name → adjacent vote number.
    """
    # If model echoed its own system prompt, there's nothing to parse
    if raw.startswith("Extract all text from the image."):
        return {}

    # Step 1: unwrap JSON if needed
    text = raw
    try:
        outer = json.loads(raw)
        if isinstance(outer, dict):
            party_names = {p["party_name"] for p in party_list}
            # If the JSON keys ARE party names → already structured correctly
            if any(k in party_names for k in outer.keys()):
                result = {}
                for p in party_list:
                    v = re.sub(r"[^\d]", "", str(outer.get(p["party_name"], "0")).translate(THAI_MAP))
                    result[p["party_name"]] = v or "0"
                return result
            # Otherwise extract all string values (HTML table content)
            text = " ".join(str(v) for v in outer.values())
    except (json.JSONDecodeError, TypeError):
        pass

    # Step 2: parse HTML table cells
    cells = re.findall(r"<td[^>]*>(.*?)</td>", text, re.DOTALL)
    cells = [re.sub(r"\s+", " ", c).strip() for c in cells]

    party_names = [p["party_name"] for p in party_list]

    def find_party_in_cell(cell: str) -> str | None:
        """Return matched party name: exact match first, then fuzzy."""
        # 1. Exact match (longest first to avoid ประชาชน matching รวมคณะประชาชน)
        exact = [p for p in party_names if p == cell.strip()]
        if exact:
            return exact[0]
        # 2. Substring: prefer longest matching party name to avoid short-name collision
        subs = [p for p in party_names if p in cell]
        if subs:
            return max(subs, key=len)
        # 3. Fuzzy fallback via difflib (catches OCR misreads like ไทยกร่างไทย→ไทยสร้างไทย)
        matches = difflib.get_close_matches(cell.strip(), party_names, n=1, cutoff=0.75)
        return matches[0] if matches else None

    votes: dict[str, str] = {}
    for i, cell in enumerate(cells):
        party = find_party_in_cell(cell)
        if party and party not in votes:
            # Look for a vote number in this cell or the next 2 cells
            for j in range(i, min(i + 3, len(cells))):
                m = re.search(r"([๐-๙][๐-๙,]*[๐-๙]|[0-9][0-9,]*[0-9])", cells[j])
                if m:
                    num = m.group(1).translate(THAI_MAP).replace(",", "")
                    if num.isdigit() and int(num) >= 1:
                        votes[party] = num
                        break

    return votes

# Single Image API Call

1. แปลงรูปเป็น Base64
   - เรียก to_base64(img_path) เพื่อส่งรูปเข้า API
2. สร้าง request content
   - ใส่รูปแบบ base64
   - ใส่ prompt ให้ model OCR

รูปแบบ
```
content = [
  image,
  "Extract all text from the image."
]
```
3. วน retry หลายรอบ ป้องกันกรณี model ตอบผิด / ว่าง / rate limit โดยใช้

```
for attempt in range(1, MAX_RETRIES + 1):
```
4. เรียก OpenAI API โดยส่ง model (เช่น GPT-4o vision) โดย message = รูป + prompt และได้ response กลับมา

```
resp = client.chat.completions.create(...)
```
5. ดึงข้อความ OCR คือ text ที่ model อ่านจากรูป

```
raw = resp.choices[0].message.content.strip()
```
6. parse ข้อมูล โดยแปลง raw text → {party: votes}

```
result = parse_votes_from_ocr(raw, party_list)
```
   - ถ้า parse สำเร็จ → return
   - ถ้า parse ไม่ได้ → retry
     - รอ RETRY_DELAY
     - ลองใหม่ (เพราะ model stochastic)
     - ถ้าครบ retry แล้วยังไม่ได้ → log warning (print("[WARN] no parties extracted") แล้วไปต่อ
   - Handling Error
     - Error 429 : รอ 65 วินาที แล้ว retry
     - Error 400 : input ผิด → return {} ทันที
     - Else : RETRY_DELAY * attempt

In [ ]:
def call_one_image(
    img_path: Path,
    party_list: list[dict],
    client: OpenAI,
) -> dict[str, str]:
    b64 = to_base64(img_path)
    content = [
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
        {"type": "text", "text": "Extract all text from the image."},
    ]
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": content}],
                max_tokens=MAX_TOKENS,
                timeout=90,
            )
            raw = resp.choices[0].message.content.strip()
            result = parse_votes_from_ocr(raw, party_list)
            if result:
                return result
            # No parties found — retry (model is stochastic)
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)
            else:
                print(f"[WARN] no parties extracted from {img_path.name}")
            continue

        except Exception as e:
            msg = str(e)
            if "429" in msg or "rate" in msg.lower():
                print(f"[RATE LIMIT] Waiting 65s...")
                time.sleep(65)
            elif "400" in msg:
                print(f"[BAD REQUEST] {e}")
                return {}
            else:
                print(f"[ERROR {attempt}/{MAX_RETRIES}] {type(e).__name__}: {e}")
                if attempt < MAX_RETRIES:
                    time.sleep(RETRY_DELAY * attempt)
    return {}

# Merge Pages

Flow

```
page1 OCR
page2 OCR
page3 OCR
   ↓
merge_pages()
   ↓
final result
```

เอา max ของแต่ละพรรคจากทุกหน้า หน้าสุดท้ายมักเป็นยอดรวม ซึ่งมากกว่ายอดย่อยรายหน่วย โดยมีขั้นตอน ดังนี้

1. รับ input pages = list ของ dict

```
[
  {"พรรคA": "100", "พรรคB": "50"},
  {"พรรคA": "120", "พรรคB": "55"},
]
```

2. รวมชื่อพรรคทั้งหมดจากทุกหน้า

```
all_keys = {k for r in pages for k in r}
```

3. วนทีละพรรค

```
for party in all_keys:
```

4. ดึงค่าของพรรคนั้นจากทุกหน้า
5. ทำความสะอาดข้อมูล: ลบ non-digit (เช่น comma, space)

```
digits = re.sub(r"[^\d]", "", ...)
```

6. แปลงเป็น int แล้วเก็บใน list

```
vals.append(int(digits))
```

7. เลือกค่ามากสุด

```
max(vals)
```

8. แปลงกลับเป็น string แล้วเก็บ ถ้าไม่มีค่าเลย → ใช้ "0"

```
merged[party] = str(max(vals))
```

9. return dict ที่ merge แล้ว


input:

```
pages = [
  {"ประชาชน": "10,000"},
  {"ประชาชน": "34,167"},
]
```

output:

```
{"ประชาชน": "34167"}
```



















In [ ]:
def merge_pages(pages: list[dict[str, str]]) -> dict[str, str]:
    """เอา max ของแต่ละพรรคจากทุกหน้า หน้าสุดท้ายมักเป็นยอดรวม ซึ่งมากกว่ายอดย่อยรายหน่วย"""
    all_keys = {k for r in pages for k in r}
    merged = {}
    for party in all_keys:
        vals = []
        for r in pages:
            digits = re.sub(r"[^\d]", "", str(r.get(party, "0")))
            if digits:
                vals.append(int(digits))
        merged[party] = str(max(vals)) if vals else "0"
    return merged

# Extract Votes for a Document

1. กรณีมีแค่ 1 รูป `if len(images) == 1:` ไม่ต้อง merge เรียก OCR เลย `call_one_image(...)`

2. เตรียม list เก็บผลแต่ละหน้า `pages_results = []`

3. วน OCR ทีละหน้า `for i, img_path in enumerate(images):`

ผลลัพธ์
```
[1/3] doc1.png
```

4. เรียก OCR ต่อหน้า `result = call_one_image(img_path, ...)`

ผลลัพธ์
```
{"พรรคA": "100", "พรรคB": "50"}
```

5. เก็บผลแต่ละหน้า `pages_results.append(result)`

6. หน่วงเวลา (กัน rate limit) ทำเฉพาะถ้ายังไม่ใช่หน้าสุดท้าย โดยใช้ `time.sleep(RATE_DELAY)`

7. รวมผลทุกหน้า ใช้ max ของแต่ละพรรค

```
merged = merge_pages(pages_results)
```

8. นับจำนวนพรรคที่มีคะแนน

```
nz = sum(1 for v in merged.values() if v != "0")
```

9. สรุปผลการสกัดข้อมูล `merged → 10/20 non-zero`

10. return ผลลัพธ์











In [ ]:
def extract_votes(
    party_list: list[dict],
    images: list[Path],
    client: OpenAI,
) -> dict[str, str]:
    if len(images) == 1:
        return call_one_image(images[0], party_list, client)

    pages_results = []
    for i, img_path in enumerate(images):
        print(f"[{i+1}/{len(images)}] {img_path.name}")
        result = call_one_image(img_path, party_list, client)
        pages_results.append(result)
        if i < len(images) - 1:
            time.sleep(RATE_DELAY)

    merged = merge_pages(pages_results)
    nz = sum(1 for v in merged.values() if v != "0")
    print(f"merged → {nz}/{len(party_list)} non-zero")
    return merged

# Checkpoint

ฟังก์ชันนี้ใช้สำหรับ โหลด checkpoint (สถานะการทำงานที่เคยบันทึกไว้) เพื่อให้สามารถ resume งานต่อได้

เช็คว่าไฟล์ CHECKPOINT มีอยู่หรือไม่

ถ้ามี:
 - อ่านไฟล์ทั้งหมดเป็น text
 - แปลงจาก JSON → Python dict
 - return dict ที่โหลดมา

ถ้าไฟล์ไม่มี:
 - return {} (dict ว่าง)

checkpoint.json

```
{
  "doc_001": {"พรรคA": "100"},
  "doc_002": {"พรรคB": "200"}
}
```

output

```
{
  "doc_001": {"พรรคA": "100"},
  "doc_002": {"พรรคB": "200"}
}
```

In [ ]:
def load_checkpoint() -> dict:
    if Path(CHECKPOINT).exists():
        return json.loads(Path(CHECKPOINT).read_text())
    return {}

**สรุปเป็น bullet**

- รับ input: results → dict ของผลลัพธ์ เช่น {doc_id: votes}
- แปลง dict → JSON string
- ensure_ascii=False ทำให้ ภาษาไทยไม่ถูกแปลงเป็น unicode escape เช่น "ประชาชน" จะยังอ่านได้ ไม่ใช่ \u0e1b...
- เขียนลงไฟล์ CHECKPOINT ถ้าไฟล์ไม่มี → สร้างใหม่ และถ้ามีอยู่แล้ว → overwrite

```
{
  "doc_001": {
    "ประชาชน": "34167"
  }
}
```

In [ ]:
def save_checkpoint(results: dict) -> None:
    Path(CHECKPOINT).write_text(json.dumps(results, ensure_ascii=False))

### ทำความสะอาดข้อมูลคะแนน (votes) ให้เหลือเฉพาะตัวเลขล้วน

สรุปเป็น bullet
- รับ input: raw → ค่าอะไรก็ได้ (string / int / None)
- ลบทุกอย่างที่ "ไม่ใช่ตัวเลข" `re.sub(r"[^\d]", "", ...)`
  - [^\d] = ทุกตัวที่ไม่ใช่ 0–9 เช่น:
    - "34,167" → "34167"
    - "๓๔,๑๖๗" (ถ้ายังไม่ map ไทย) → อาจหายหมดหรือผิด ต้องแปลงก่อน
    - "votes: 120" → "120"
  - เก็บผลไว้ใน digits
  - ถ้าไม่มีตัวเลขเลย → return "0"

```
clean_votes("34,167")     → "34167"
clean_votes(" 120 คน ")   → "120"
clean_votes(None)         → "0"
clean_votes("")           → "0"
```



In [ ]:
def clean_votes(raw) -> str:
    digits = re.sub(r"[^\d]", "", str(raw))
    return digits or "0"

# Execute Extraction OCR

In [ ]:
from google.colab import userdata
import os
os.environ["TYPHOON_API_KEY"] = userdata.get("TYPHOON_API_KEY")


**Flow**

```
template.csv
    ↓
group by doc
    ↓
for each doc:
    images → OCR → parse → clean
    ↓
    save checkpoint
    ↓
merge all → submission.csv
```



1. Config

- RESET → ล้าง checkpoint หรือไม่
- LIMIT → จำกัดจำนวน document ที่จะรัน

2. Reset checkpoint (ถ้าเปิด)

```
if RESET:
    Path(CHECKPOINT).unlink(...)
```
ลบไฟล์ checkpoint → เริ่มใหม่ทั้งหมด

3. โหลด template

```
df, doc_parties = load_template()
```

- df → ตารางทั้งหมด
- doc_parties → group ตาม doc_id

4. โหลด checkpoint เก็บผลที่ทำไปแล้ว

```
results = load_checkpoint()
```

5. ฟังก์ชันเช็คว่า doc เสร็จยัง ถ้า id ของทุก party อยู่ใน results → ถือว่า done

```
def done(doc_id):
```

6. หา document ที่ยังไม่ทำ โดย filter เฉพาะ doc ที่ยังไม่เสร็จ จำกัดจำนวน (ถ้ามี LIMIT)

```
remaining = [...]
```

7. Loop ทีละ document

```
for i, doc_id in enumerate(remaining):
```

   - 7.1 โหลดรูป `images = get_doc_images(doc_id)`
   - 7.2 ดึง party list `party_list = doc_parties[doc_id]`
   - 7.3 print progress `[1/10] doc_001 (3 pages, 20 parties)`
   - 7.4 ถ้าไม่มีรูป `if not images:`
     - ตั้งค่า vote ทุกพรรค = "0"
     - save checkpoint
     - continue
   - 7.5 OCR + Extract
     `votes = extract_votes(...)` ได้ {party_name: votes}
   - 7.6 Clean & map → results `for p in party_list:` ใช้ clean_votes() และ map ค่า party_name → id
   - 7.7 Save checkpoint `save_checkpoint(results)` โดย save ทุก doc (กัน crash)
   - 7.8 หน่วงเวลา `time.sleep(RATE_DELAY)` กัน rate limit

8. Export CSV (Submission)
   - สร้าง list ของ rows `rows = [{"id": ..., "votes": ...}]`
   - แปลงเป็น DataFrame `sub = pd.DataFrame(rows)`
   - save เป็น submission.csv `sub.to_csv(OUTPUT_FILE)`

9. สรุปผล
   - นับจำนวน vote ที่ไม่ใช่ 0 `nz = (sub["votes"] != "0").sum()`

In [ ]:
RESET = False  # Set to True to clear the checkpoint
LIMIT = 1   # Set to an integer to limit the number of documents processed

api_key = os.environ.get("TYPHOON_API_KEY")
client = OpenAI(api_key=api_key, base_url=BASE_URL)

if RESET:
    Path(CHECKPOINT).unlink(missing_ok=True)
    print("Checkpoint cleared.")

df, doc_parties = load_template()
print(f"{len(df)} rows | {len(doc_parties)} documents")

results = load_checkpoint()
print(f"Checkpoint: {len(results)} rows done")

def done(doc_id): return all(p["id"] in results for p in doc_parties[doc_id])

remaining = [d for d in sorted(doc_parties) if not done(d)]
if LIMIT:
    remaining = remaining[: LIMIT]
print(f"Remaining : {len(remaining)} documents\n")

for i, doc_id in enumerate(remaining):
    images     = get_doc_images(doc_id)
    party_list = doc_parties[doc_id]
    print(f"[{i+1}/{len(remaining)}] {doc_id}  ({len(images)}p, {len(party_list)} parties)")

    if not images:
        for p in party_list:
            results[p["id"]] = "0"
        save_checkpoint(results)
        continue

    votes = extract_votes(party_list, images, client)

    matched = 0
    for p in party_list:
        v = clean_votes(votes.get(p["party_name"], "0"))
        results[p["id"]] = v
        if v != "0":
            matched += 1

    print(f"Non-zero: {matched}/{len(party_list)}")
    save_checkpoint(results)
    time.sleep(RATE_DELAY)

# Write output
rows = [{"id": r["id"], "votes": results.get(r["id"], "0")} for _, r in df.iterrows()]
sub = pd.DataFrame(rows)
sub.to_csv(OUTPUT_FILE, index=False)
nz = (sub["votes"] != "0").sum()
print(f"\nSaved {OUTPUT_FILE}: {len(sub)} rows | {nz} non-zero ({nz/len(sub)*100:.1f}%) errors")